In [ ]:
#Install necessary packages
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity


In [21]:
#Setting up environment variables
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool


load_dotenv()

foundry_project_endpoint =os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

In [22]:
#Setting up the project client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

In [ ]:
openai_client= project_client.get_openai_client()


In [ ]:
#Getting the MCP Server connection ID
connection_id=""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"MCP Server Connection ID: {connection_id}")

In [ ]:
#Creating the MCP Tool Spec
tool=MCPTool(
    server_label="microsoft-learn-mcp-server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)


In [ ]:
agent = project_client.agents.create_version(
    agent_name="mcp-server-agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        tools=[tool],
        instructions="You are an assistant that can answer questions about Microsoft Learn content using the Microsoft Learn MCP Server tool.",
    )
)

print(f"Created agent with ID: {agent.id} and name: {agent.name}")

In [ ]:
conversation = openai_client.conversations.create()
print(f"Created conversation with ID: {conversation.id}")

In [ ]:
# user input
user_message = "can you provide a boiler c# code for http azure function?"

In [ ]:
response= openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent":{
            "name":"mcp-server-agent",
            "type":"agent_reference"
        }
    },
    input=user_message
    
)

print(f"Agent response: {response.output_text}")